**Azure Blob Storage** is Azure's object storage service — massively scalable, highly durable, and foundational to virtually every Azure architecture. It stores unstructured data (files, images, videos, backups, logs, data lake files) as objects called **blobs** inside containers, accessed over HTTP/HTTPS.

Blob Storage is the Azure equivalent of AWS S3. The structure is: **Storage Account → Container → Blob** (vs S3's Account → Bucket → Object).

## Storage Accounts

A **Storage Account** is the top-level namespace for all Azure Storage services — Blob, Files, Queues, and Tables. Every storage account has a unique name that forms part of the endpoint URL.

```
https://<account-name>.blob.core.windows.net/<container>/<blob>
```

### Account types

| Type | Use case |
|---|---|
| **Standard general-purpose v2 (GPv2)** | Default choice — supports Blob, Files, Queues, Tables; all access tiers |
| **Premium Block Blob** | High-throughput, low-latency blob workloads (SSD-backed); no access tier support |
| **Premium Page Blob** | Azure VM unmanaged disks (legacy); random read/write workloads |
| **Premium File Shares** | High-performance Azure Files (SMB/NFS) |

> For almost all new workloads use **Standard GPv2** — it supports all blob features including lifecycle management and access tiers.

## Redundancy Options

Azure Storage redundancy is configured per storage account and determines how many copies of your data exist and where.

| Option | Copies | Where | Durability | Use case |
|---|---|---|---|---|
| **LRS** (Locally Redundant) | 3 | Same data centre | 11 nines | Dev/test; data that can be recreated |
| **ZRS** (Zone Redundant) | 3 | 3 zones in same region | 12 nines | Production; survives zone failure |
| **GRS** (Geo-Redundant) | 6 | Primary (LRS) + secondary region (LRS) | 16 nines | DR; secondary is read-only unless failover |
| **GZRS** (Geo-Zone-Redundant) | 6 | Primary (ZRS) + secondary region (LRS) | 16 nines | Highest availability + DR |
| **RA-GRS** | 6 | Same as GRS | 16 nines | GRS + read access to secondary at all times |
| **RA-GZRS** | 6 | Same as GZRS | 16 nines | GZRS + read access to secondary at all times |

### Choosing redundancy

```
Need DR (survive region failure)?  → GRS / GZRS / RA-GRS / RA-GZRS
  └── Also need read from secondary without failover? → RA-GRS / RA-GZRS
  └── No DR needed:
        Need to survive zone failure? → ZRS  (recommended for production)
        Single datacentre is fine?    → LRS  (cheapest)
```

> **RA-GRS secondary endpoint:** `https://<account>-secondary.blob.core.windows.net` — use for read-only access during a primary region outage.

## Containers and Blobs

### Containers

A **container** organises blobs — equivalent to an S3 bucket, but nested inside a storage account.

- Names: 3–63 characters, lowercase letters, numbers, and hyphens
- A storage account can have unlimited containers
- Public access can be set at the container level: **Private** (default), **Blob** (anonymous read on blobs), or **Container** (anonymous list + read)

### Blob types

| Type | Description | Use case |
|---|---|---|
| **Block Blob** | Composed of blocks; optimised for sequential upload | Files, images, videos, backups, data lake |
| **Append Blob** | Blocks can only be appended — not modified or deleted | Log files, audit trails, streaming data |
| **Page Blob** | 512-byte pages; supports random read/write | Azure VM unmanaged disks (legacy) |

Block Blob is the default and the type you will use for almost everything. Managed Disks have replaced Page Blobs for VM storage.

### Blob size limits (Block Blob)

| Operation | Limit |
|---|---|
| Single PUT (small upload) | Up to 5,000 MB |
| Block size | Up to 4,000 MB per block |
| Max blob size | 190.7 TiB |
| Max blocks per blob | 50,000 |

Use **Put Block + Put Block List** (parallel block upload) for large files — analogous to S3 multipart upload.

## Access Tiers

Access tiers balance storage cost against access cost. Set at the account level (default) or overridden per blob.

| Tier | Storage cost | Access cost | Min duration | Retrieval time | Use case |
|---|---|---|---|---|---|
| **Hot** | Highest | Lowest | None | Immediate | Frequently accessed data |
| **Cool** | Lower | Higher | 30 days | Immediate | Infrequent access, backups kept online |
| **Cold** | Even lower | Higher | 90 days | Immediate | Rarely accessed, still online |
| **Archive** | Lowest | Highest | 180 days | Hours (rehydration) | Long-term retention, compliance |

### Archive tier — rehydration

Blobs in the Archive tier are **offline** — they cannot be read directly. To access an archived blob you must first **rehydrate** it by changing its tier to Hot, Cool, or Cold:

| Rehydration priority | Time |
|---|---|
| **Standard** | Up to 15 hours |
| **High** | Under 1 hour (higher cost) |

> You are billed for the minimum duration even if you delete the blob early. Tiering down (Hot → Cool → Archive) costs nothing; tiering up (Archive → Hot) incurs a rehydration read cost.

## Lifecycle Management

**Lifecycle management policies** automatically transition blobs between access tiers or delete them based on age or last-access time — reducing storage costs without manual intervention.

```json
{
  "rules": [{
    "name": "tiering-rule",
    "type": "Lifecycle",
    "definition": {
      "filters": { "blobTypes": ["blockBlob"], "prefixMatch": ["logs/"] },
      "actions": {
        "baseBlob": {
          "tierToCool": { "daysAfterModificationGreaterThan": 30 },
          "tierToArchive": { "daysAfterModificationGreaterThan": 90 },
          "delete": { "daysAfterModificationGreaterThan": 365 }
        },
        "snapshot": { "delete": { "daysAfterCreationGreaterThan": 30 } }
      }
    }
  }]
}
```

Filters can match by blob type, prefix, blob index tags, or blob size. Actions include tier transition and delete — applied to the base blob, snapshots, and/or versions independently.

## Security and Access Control

### Authentication methods

| Method | Recommended? | Notes |
|---|---|---|
| **Entra ID + Azure RBAC** | ✅ Yes | Fine-grained roles; full audit trail; Managed Identity support |
| **User Delegation SAS** | ✅ Yes | Time-limited URL; signed with Entra ID credentials — not account key |
| **Shared Access Signature (SAS) — Account/Service** | ⚠️ With care | Signed with account key — key compromise = full account compromise |
| **Account Key** | ❌ Avoid | Full admin access; treat like a root password; rotate regularly |

### Azure RBAC roles for Blob Storage

| Role | Access |
|---|---|
| **Storage Blob Data Owner** | Full control — read, write, delete, set access control |
| **Storage Blob Data Contributor** | Read, write, delete blobs |
| **Storage Blob Data Reader** | Read blobs only |
| **Storage Blob Delegator** | Create User Delegation SAS tokens |

### Shared Access Signatures (SAS)

A **SAS token** is a query string appended to a blob URL that grants time-limited, scoped access — without exposing account credentials.

```
https://myaccount.blob.core.windows.net/images/photo.jpg
  ?sv=2023-01-03
  &se=2025-12-31T00%3A00%3A00Z     ← expiry
  &sr=b                             ← scope: blob
  &sp=r                             ← permissions: read
  &sig=<signature>
```

| SAS type | Signed with | Best for |
|---|---|---|
| **User Delegation SAS** | Entra ID credentials | Always prefer — revocable, no account key exposure |
| **Service SAS** | Account key | Legacy; limited to one storage service |
| **Account SAS** | Account key | Legacy; spans multiple services |

### Stored Access Policies

Define reusable SAS policies on a container. Update or revoke the policy to instantly invalidate all SAS tokens derived from it — without needing to rotate the account key.

### Network security

- **Firewall rules** — restrict access to specific IP ranges or VNet subnets
- **Service endpoints** — route traffic from a VNet subnet to storage over the Azure backbone, not the internet
- **Private endpoints** — assign the storage account a private IP within your VNet; DNS resolves to the private IP; no public internet exposure

## Encryption

### Encryption at rest

All data in Azure Storage is **encrypted at rest by default** using 256-bit AES — no configuration needed.

| Key type | Who manages | Notes |
|---|---|---|
| **Microsoft-managed keys** | Microsoft | Default; automatic rotation |
| **Customer-managed keys (CMK)** | You, via Key Vault | Control rotation schedule; supports BYOK; audit via Key Vault logs |
| **Customer-provided keys** | You, per request | You pass the key with each API call; Azure never stores it |

### Encryption in transit

Enable **Secure transfer required** on the storage account to enforce HTTPS for all requests. HTTP requests are rejected. This is recommended and is the default for new accounts.

### Infrastructure encryption

Enable a second layer of AES-256 encryption at the infrastructure level for compliance scenarios that require double encryption (e.g. government, financial).

## Data Protection Features

### Blob Versioning

Automatically preserves previous versions of a blob when it is overwritten or deleted. Access previous versions by their version ID.

```
PUT report.csv  →  version 2025-01-01T10:00Z  (current)
                   version 2025-01-01T09:00Z  (previous)
                   version 2025-01-01T08:00Z
```

### Soft Delete

Deleted blobs (and containers) are retained for a configurable period (1–365 days) and can be undeleted. Different from versioning — soft delete captures explicit deletions; versioning captures overwrites.

| Feature | Protects against | Recovery |
|---|---|---|
| Soft delete | Accidental deletion | Undelete within retention period |
| Versioning | Overwrites and deletions | Access previous version by ID |
| Point-in-time restore | Bulk accidental changes | Restore all blobs to a prior timestamp |

### Immutable Storage (WORM)

**WORM (Write Once, Read Many)** policies prevent any modification or deletion of blobs for a specified period — enforced at the storage account or container level.

| Policy type | Description |
|---|---|
| **Time-based retention** | Blobs cannot be modified or deleted until the retention period expires |
| **Legal hold** | Blobs are locked until the legal hold is explicitly removed — no fixed duration |

Once a time-based policy is **locked**, the retention period cannot be shortened (only extended) — even by a subscription admin. This satisfies SEC 17a-4 and other regulatory requirements.

## Performance and Large Uploads

### Parallel block upload

For large blobs, upload multiple blocks in parallel then commit them with a block list:

```
Put Block (block 1) ─┐
Put Block (block 2) ─┤ → Put Block List → assembled blob
Put Block (block 3) ─┘
```

The Python SDK (`BlobClient.upload_blob`) handles this automatically with configurable `max_concurrency` and `max_single_put_size`.

### Object replication

**Object replication** asynchronously copies blobs from a source container to a destination container — across storage accounts, subscriptions, or regions. Combined with lifecycle management on the destination, you can build cost-optimised cross-region archive pipelines.

### Static website hosting

Enable static website hosting on a storage account to serve HTML, CSS, and JavaScript directly from Blob Storage:

```
https://<account>.z13.web.core.windows.net/
```

Pair with **Azure CDN** or **Azure Front Door** for:
- Custom domain with HTTPS
- Global CDN caching
- WAF protection
- Keeping the storage account private (traffic only from CDN)

## Working with Blob Storage via Python

In [ ]:
# pip install azure-identity azure-storage-blob azure-mgmt-storage

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.storage.blob import (
    BlobServiceClient, BlobClient, ContainerClient,
    generate_blob_sas, BlobSasPermissions, UserDelegationKey
)
from datetime import datetime, timezone, timedelta

credential = DefaultAzureCredential()
account_name = "mystorageaccount"
account_url  = f"https://{account_name}.blob.core.windows.net"

blob_service = BlobServiceClient(account_url=account_url, credential=credential)

In [ ]:
# Create a container
container = blob_service.create_container("my-container")
print(f"Container created: {container.container_name}")

# List all containers
for c in blob_service.list_containers():
    print(f"  {c.name}")

In [ ]:
from azure.storage.blob import StandardBlobTier

container_client = blob_service.get_container_client("my-container")

# Upload a small blob (auto single PUT)
container_client.upload_blob(
    name="data/config.json",
    data=b'{"env": "production"}',
    overwrite=True
)

# Upload a large file with parallel blocks
blob_client = container_client.get_blob_client("data/large-file.bin")
with open("large-file.bin", "rb") as f:
    blob_client.upload_blob(
        f,
        overwrite=True,
        max_concurrency=4,          # 4 parallel block uploads
        max_single_put_size=4 * 1024 * 1024  # single PUT up to 4 MB
    )
print("Upload complete")

In [ ]:
# List blobs with their size and tier
for blob in container_client.list_blobs(include=["metadata"]):
    tier = blob.blob_tier or "N/A"
    size_kb = blob.size // 1024
    print(f"{blob.name:<50} {size_kb:>8} KB  tier: {tier}")

In [ ]:
from azure.storage.blob import StandardBlobTier

# Move a blob to the Archive tier
blob_client = container_client.get_blob_client("data/old-report.csv")
blob_client.set_standard_blob_tier(StandardBlobTier.ARCHIVE)
print("Moved to Archive")

# Rehydrate from Archive to Hot (high priority — < 1 hour)
from azure.storage.blob import RehydratePriority
blob_client.set_standard_blob_tier(
    StandardBlobTier.HOT,
    rehydrate_priority=RehydratePriority.HIGH
)
print("Rehydrating to Hot (high priority)")

In [ ]:
# Generate a User Delegation SAS (signed with Entra ID — recommended)
start  = datetime.now(timezone.utc)
expiry = start + timedelta(hours=2)

# Get a user delegation key (valid for up to 7 days)
udk: UserDelegationKey = blob_service.get_user_delegation_key(start, expiry)

sas_token = generate_blob_sas(
    account_name=account_name,
    container_name="my-container",
    blob_name="data/config.json",
    user_delegation_key=udk,
    permission=BlobSasPermissions(read=True),
    expiry=expiry
)

sas_url = f"{account_url}/my-container/data/config.json?{sas_token}"
print(f"SAS URL (valid 2h):\n{sas_url}")

In [ ]:
# List blob versions (requires versioning enabled on the account)
for blob in container_client.list_blobs(include=["versions"]):
    if blob.version_id:
        current = "← current" if blob.is_current_version else ""
        print(f"{blob.name:<40} version: {blob.version_id}  {current}")

## Summary

| Concept | Key Takeaway |
|---|---|
| Storage account | Top-level namespace — endpoint `<account>.blob.core.windows.net`; use Standard GPv2 |
| LRS | 3 copies in one data centre — cheapest; dev/test only |
| ZRS | 3 copies across 3 zones — recommended for production |
| GRS / GZRS | Zone or local redundancy + async copy to a secondary region — for DR |
| RA-GRS / RA-GZRS | GRS/GZRS + read access to secondary without failover |
| Block Blob | Default blob type — general file storage; parallel block upload for large files |
| Append Blob | Append-only — log files and streaming data |
| Page Blob | Random read/write — legacy VM disks; replaced by Managed Disks |
| Hot tier | Frequent access — highest storage cost, lowest access cost |
| Cool / Cold tier | Infrequent — lower storage cost; 30/90-day minimum |
| Archive tier | Offline — rehydrate before reading; up to 15h standard, < 1h high priority |
| Lifecycle management | Auto-transition or delete blobs by age or last-access — JSON policy rules |
| Entra ID + RBAC | Recommended auth — fine-grained; Managed Identity for services |
| User Delegation SAS | Time-limited URL signed with Entra ID credentials — prefer over account-key SAS |
| Stored Access Policy | Reusable SAS rules on a container — revoke without rotating the account key |
| Soft delete | Retain deleted blobs for 1–365 days — recoverable |
| Versioning | Preserve every overwrite — access previous versions by ID |
| Immutable storage (WORM) | Time-based retention or legal hold — cannot be shortened once locked |
| Private endpoint | Private IP for storage in your VNet — no public internet exposure |